In [ ]:
import kagglehub
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from tqdm import tqdm

%matplotlib inline
clear_output()


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

csv_path = os.path.join(path,"Q3_data.csv")

df = pd.read_csv(csv_path)


In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:

missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})

missing_data = missing_data[missing_data['Missing_Percentage'] > 30].sort_values('Missing_Percentage', ascending=False)
missing_data.head(50)

In [ ]:

#biger than 30%
print(f"Before: {df.shape}")
df = df.drop(missing_data['Column'],axis=1)
print(f"After dropping missing price/year/odometer: {df.shape}")

for i in df.columns:
    df[i] = df[i].fillna(df[i].mode()[0])



print("Missing values remaining:", df.isnull().sum().sum())
print(f"After dropping missing price/year/odometer: {df.shape}")


In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 3: Write your code here:

from sklearn.preprocessing import LabelEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

le = LabelEncoder()
df[list(categorical_cols)] = le.fit_transform(list(categorical_cols))


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler
feature = df.drop("Target",axis=1).columns


standard_scaler = StandardScaler()
df[feature] = standard_scaler.fit_transform(df[feature])

df.head(5)

In [ ]:
# Task 5: Write your code here:

# 1. Is the target imbalanced?
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

print("0.736563 not imblanced")

In [ ]:
# Task 1: Write your code here:
X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

n_splits = 5 # K=5 Folds

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

sklearn_models = {

  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'f1': []}

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  # 1. Split data
  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # 2. Train & Validate sklearn models
  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train) # train
    y_pred = model.predict(X_test) # validate

    # 3. Save metrics for that model in this fold
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)
    all_results[model_name]['f1'].append(f1)

print("________________________________")
print()
for model_name in all_results:
  print(f"\n{model_name}:")
  print(f"  Accuracy:  {np.mean(all_results[model_name]['accuracy']):.4f}")
  print(f"  F1-Score:  {np.mean(all_results[model_name]['f1']):.4f}")

In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_



for i, (model_name, imp) in enumerate(importances.items()):
  plt.figure( figsize=(18, 6))
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  plt.barh(features[sorted_idx], imp[sorted_idx])
  plt.title(f"{model_name} Feature Importance")
  plt.xlabel("Importance Score")

plt.tight_layout()
plt.show()

In [ ]:
print(features[sorted_idx])

In [ ]:
# Task 2: Write your code here:
print(f"The importances featureis: P_2")

In [ ]:
# Task Bonus: Write your code here: